In [ ]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from pytrends.request import TrendReq

load_dotenv(".env")
TMDB_TOKEN = os.getenv("TMDB_TOKEN")

BASE_URL = "https://api.themoviedb.org/3"
GEO      = "DE"

tmdb_headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_TOKEN}"
}

pytrends = TrendReq(hl="de-DE", tz=60)

## Getter

In [ ]:
def tmdb_get(endpoint, params=None, retries=3):
    if params is None:
        params = {}
    url = BASE_URL + endpoint
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=tmdb_headers, params=params, timeout=10)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.Timeout:
            print(f"Timeout, retry {attempt + 1}/{retries}...")
            time.sleep(2)
        except requests.exceptions.RequestException as e:
            print(f"Fehler: {e}")
            time.sleep(2)
    return None

## Collect Movies

In [ ]:
def collect_movies(year, pages=5):
    all_movies = []
    for page in range(1, pages + 1):
        data = tmdb_get(
            "/discover/movie",
            params={
                "primary_release_year": year,
                "sort_by":              "popularity.desc",
                "vote_count.gte":       500,
                "page":                 page,
            }
        )
        all_movies.extend(data["results"])
        time.sleep(0.25)
    return pd.DataFrame(all_movies)

frames = []
for year in range(2010, 2024):
    frames.append(collect_movies(year, pages=5))
    print(f"Jahr {year} geladen")

df_movies = pd.concat(frames, ignore_index=True).drop_duplicates("id")
print(f"Gesamt: {len(df_movies)} Filme")

## Load Details

### Trends before

In [ ]:
def get_trends_before(title, release):
    start = (release - pd.DateOffset(months=6)).strftime("%Y-%m-%d")
    end   = release.strftime("%Y-%m-%d")
    try:
        pytrends.build_payload(kw_list=[title], timeframe=f"{start} {end}", geo=GEO)
        df = pytrends.interest_over_time()
        return {
            "avg_trend_before":  round(df[title].mean(), 2) if not df.empty else None,
            "max_trend_before":  df[title].max()            if not df.empty else None,
            "peak_date_before":  df[title].idxmax()         if not df.empty else None,
        }
    except:
        return {"avg_trend_before": None, "max_trend_before": None, "peak_date_before": None}

### Trends After

In [ ]:
def get_trends_after(title, release):
    start = release.strftime("%Y-%m-%d")
    end   = (release + pd.DateOffset(months=1)).strftime("%Y-%m-%d")
    try:
        pytrends.build_payload(kw_list=[title], timeframe=f"{start} {end}", geo=GEO)
        df = pytrends.interest_over_time()
        return {
            "avg_trend_after": round(df[title].mean(), 2) if not df.empty else None,
            "max_trend_after": df[title].max()            if not df.empty else None,
        }
    except:
        return {"avg_trend_after": None, "max_trend_after": None}

## Load Details

In [ ]:
rows = []

for _, movie in df_movies.iterrows():
    title   = movie["title"]
    release = pd.to_datetime(movie["release_date"])

    details = tmdb_get(f"/movie/{movie['id']}")
    if details is None:
        continue

    budget  = details.get("budget",  0)
    revenue = details.get("revenue", 0)

    if budget == 0 or revenue == 0:
        print(f"  skipped: {title}")
        continue

    before = get_trends_before(title, release)
    time.sleep(1)
    after  = get_trends_after(title, release)
    time.sleep(1)

    rows.append({
        "title":        title,
        "release_date": details.get("release_date"),
        "budget":       budget,
        "revenue":      revenue,
        "roi":          round(revenue / budget, 2),
        "vote_average": details.get("vote_average"),
        "vote_count":   details.get("vote_count"),
        "runtime":      details.get("runtime"),
        "genres":       ", ".join(g["name"] for g in details.get("genres", [])),
        **before,
        **after,
    })

    print(f"Geladen: {title}")

print(f"Filme gesamt: {len(rows)}")

## Store

In [ ]:
df = pd.DataFrame(rows)
df.to_csv("Q9.csv", index=False)
print(f"Gespeichert: {len(df)} Filme")
df.head()